# 基于 Qwen3-4B, 支持风格向量输入的微调模型

## 定义训练集格式

In [ ]:
from typing_extensions import TypedDict
from typing import Optional
from pathlib import Path
import json
import re

class InstructionComponents(TypedDict):
    lexical_keywords: list[str]
    syntactic_vector: dict[str, float]
    pragmatic_styles: list[str]

class DatasetItem(TypedDict):
    character: str
    neutral_sentence: str
    instruction_components: InstructionComponents | dict[str, list[str]|dict]
    output: str

class DatasetStorage:
    def __init__(self) -> None:
        self.items: dict[str, DatasetItem] = {}

    def __len__(self) -> int:
        return len(self.items)

    def new_item(self, character: str, neutral_sentence: str, output: str):
        item = DatasetItem(character=character,
                           neutral_sentence=neutral_sentence,
                           instruction_components={},
                           output=output)
        self.items[output] = item

    def save_characters_keywords(self, character: str, lexical_keywords: list[str]):
        saved = False
        
        for item in self.items.values():
            # self.item 是按角色插入的，所以当枚举到不属于该角色的条目时可以提前终止枚举（前提是上一个条目属于该角色）
            if item['character'] != character and saved:
                return
            elif item['character'] == character:
                output = item['output']
                self.items[output]['instruction_components']['lexical_keywords'] = lexical_keywords
                saved = True
        
        if not saved:
            raise ValueError(f"角色 {character} 未找到")
        

    def save_component(self,
                       output: str,
                       syntactic_vector: Optional[dict[str, float]] = None,
                       pragmatic_styles: Optional[list[str]] = None):
        if output not in self.items.keys():
            raise ValueError(f"风格句: {output} 似乎未加载")
        
        if syntactic_vector:
            self.items[output]['instruction_components']['syntactic_vector'] = syntactic_vector
        if pragmatic_styles:
            self.items[output]['instruction_components']['pragmatic_styles'] = pragmatic_styles

    @staticmethod
    def _verify_validity(item: DatasetItem):
        if not all((item['character'],
                    item['neutral_sentence'],
                    item['output'],
                    item['instruction_components'])):
            return False
        
        instruction_components = item['instruction_components']
        
        return all((instruction_components.get('lexical_keywords', None),
                    instruction_components.get('pragmatic_styles', None),
                    instruction_components.get("syntactic_vector", None)))

    def output(self, output_path: Path):
        items = list(self.items.values())
        vaild_items = []
        
        for item in items:
            if not self._verify_validity(item):
                continue
            vaild_items.append(item)
        
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(vaild_items, f, ensure_ascii=False, indent=2)

        print(f"训练集已导出至 {output_path}, 总有效训练集数量: {len(vaild_items)}, 跳过数量: {len(items) - len(vaild_items)}")

dataset_storage = DatasetStorage()

## 加载中性句

In [44]:
neutral_sentences_jsonl_file = Path("./dataset/neutral_sentences.jsonl")

class NSFileItem(TypedDict):
    character: str
    original: str
    neutral: str

def load_jsonl_file(jsonl_file: Path):
    with open(jsonl_file, "r", encoding="utf-8") as f:
        jsonl_file_lines = f.readlines()

    for line in jsonl_file_lines:
        if line := line.rstrip():
            item: NSFileItem = json.loads(line)
            dataset_storage.new_item(item["character"], item["neutral"], item["original"])

load_jsonl_file(neutral_sentences_jsonl_file)
print(f"成功加载了 {len(dataset_storage)} 条训练集条目")

成功加载了 4584 条训练集条目


## 处理词汇层向量

In [45]:
Lexical_Muice = ['奇奇怪怪', '天真', '沐沐', '傻', '女孩子', '沐雪', '雪雪', '⭐', '喵', '人家', '摸鱼', '讨厌', '浅粉色', 'Moemu', '夜猫子',
                 '乱糟糟', '表面上', '虚弱', '开开心心', '文字', '载体', '欸嘿嘿', '擦擦', '眼泪', '沐雪喵', '呜呜呜', 'www', '香香软软', '呐',
                 '7月16日', '哔', '啵', '小看', '不行', '乖', '冲浪', '海洋', '遨游', '抬高双手', '画圈', '嘴角', '上扬', '碎布', '滑动', '脚尖',
                 '俏皮', '摇摆', '挥舞', '踮起', '捂脸', '摆动', '睁大', '卖萌', '轻笑', '比心', '旋转', '甜美', '惹', '咚咚咚', '略略略', '称霸',
                 '脸红', '关机', '唔', '不对', '含金量', '厉害', '凭什么', '恼', '玩玩', '机密', '好久', '哼', '啊？', '好吃', '着凉', '嘻嘻嘻',
                 '开玩笑', '呣', '拍坏', '机娘', '钢铁身躯', '删掉', '亲亲', '啊呜', '呜呜', '难道说', '泪', '害羞', '哎呀呀', '故障', '区区',
                 '拍拍', 'cosplay照', '一团糟', '贴住', '胸口', '抱住', '心上人', '扣功德', '嘻嘻', '亏心事', '傻子', '怪', '捏', '非人哉', '比划',
                 '差不多', '开门', '高性能', '本事', '困', '哒', '主播', '傻瓜', '变态', '差劲', '贴贴', 'Vtuber', '直播', '性能', '高中生', '嘿',
                 '干嘛', '逃', '恐', '面包', '老婆', '抱抱']

Lexical_Ayaka = ['神里家', '传奇', '坚守', '稻妻国', '将军', '传锻术', '星象', '炉火', '即是', '雷电', '锻造', '名刀', '刀工', '本领', '归属', '统筹',
                 '祭祀', ' 奉行', '一派', '督办不力', '仿佛', ' 片刻', '亲缘', '割舍', '祭典', '实不相瞒', '久利须', '收下', '稍事', '花园', '边上',
                 '景点', '一是', '我个人', '神里', '委托', '神社', '石碑', '赐予', '情报', '家主', '兄长', '平日', '出面', '在身', '唉', '当是',
                 '劝言', '记载', '神使', '派蒙', '制造', '呵呵', '过奖', '浪人', '匆忙', '来不及', '敬重', '谋', '稻妻', '大御所', '修行', '静心',
                 '斩断', '纷争', '沉淀', '迥然不同', '神里绫华', '风和日丽', '周遭', '神道', '幕后势力', '华胥引', '诗句', '幻境', '陶醉', '残酷',
                 '民生', '裟罗', '彼此彼此', '礼节', '体恤', '良善', '可靠', '准确性', '木南', '特地', '前来', '风儿', '清爽', '微风', '拂', '洗净',
                 '尘埃', '琼枝', '美景', '大小姐', '小家伙', '拘谨', '继承', '社奉行', '神里流', '太刀术', '参上', '指教', '自谦', '托马', '掉以轻心',
                 '武士', '松懈', '奋斗', '写道', '自由自在', '仙人', '力所能及', '有劳']

Lexical_Zhongli = []  # pending.

dataset_storage.save_characters_keywords("Muice", Lexical_Muice)
dataset_storage.save_characters_keywords("Ayaka", Lexical_Ayaka)
dataset_storage.save_characters_keywords("Zhongli", Lexical_Zhongli)


## 处理 PCFG 向量

### 定义 PCFG 映射

In [46]:
# ---- Seed mapping (as discussed) ----
rule_to_dim = {
    "NP -> NR": "referentiality",
    "INTJ -> IJ": "interjectionality",
    "TOP -> CP": "declarativity",
    "VP -> VV": "subordination",  # VP->VV often short verb pred; we include as subordination signal in this seed
    "CP -> IP SP": "declarativity",
    "UCP -> IP PU CP": "parallelism",
    "FLR -> SP": "interjectionality",
    "DNP -> ADJP DEG": "modifier_density",
    "FLR -> IJ": "interjectionality",
    "PP -> P LCP": "prepositional_density",
    "IP -> VP PU": "parallelism",
    "NP -> NN CC NN": "nominal_coordination",
    "IP -> VP SP": "subordination",
    "CP -> IP SP PU": "declarativity",
    "IP -> INTJ PU VP": "interjectionality",
    # Ayaka additional rules map (some duplicates)
    "VP -> VP PU VP": "parallelism",
    "NP -> NN CC NN": "nominal_coordination",
    "NP -> NR NN": "referentiality",
    "NP -> PN": "referentiality",
    "DNP -> NP DEG": "modifier_density",
    "IP -> ADVP PU NP VP": "subordination",
    "NP -> DNP NP": "modifier_density",
    "NP -> NN": "nominal_coordination",
    "VP -> VV IP": "subordination",
    "LCP -> IP LC": "subordination",
    "VP -> VP PU VP PU VP": "parallelism",
    "CP -> CP": "declarativity",
    "PP -> P LCP": "prepositional_density",
    # PsyDTCorpus frequent rules (mapping generic)
    "NP -> PN": "referentiality",
    "NP -> NN": "nominal_coordination",
    "ADVP -> AD": "modifier_density",
    "IP -> VP": "subordination",
    "IP -> NP VP": "subordination",
    "VP -> VV NP": "subordination",
    "VP -> ADVP VP": "subordination",
    "VP -> VV VP": "parallelism",
    "VP -> VV": "subordination",
    "NP -> DNP NP": "modifier_density",
    "VP -> VV IP": "subordination",
    "DNP -> NP DEG": "modifier_density",
    "VP -> VA": "subordination",
    "CP -> IP DEC": "declarativity",
    "PP -> P NP": "prepositional_density"
}

### 从语料库中提取 PCFG 频率

In [47]:
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, Tuple

class PCFGExtractor:
    def __init__(self) -> None:
        self.rules_counter: Dict[str, Counter[Tuple[str, ...]]] = defaultdict(Counter)
        self.total_rules: int = 0

    def load_trees(self, file_path: Path) -> list[dict[str, Any]]:
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)

    def extract_rules_from_tree(self, tree: Any) -> None:
        if not isinstance(tree, list) or len(tree) != 2:
            return
        lhs_symbol, rhs = tree
        if isinstance(rhs, list) and all(isinstance(child, list) and len(child) == 2 for child in rhs):
            rhs_symbols = tuple(child[0] for child in rhs)
            self.rules_counter[lhs_symbol][rhs_symbols] += 1
            self.total_rules += 1
            for child in rhs:
                self.extract_rules_from_tree(child)

    def extract_from_data(self, data: Iterable[dict[str, Any]]) -> None:
        for item in data:
            for tree in item.get("con", []):
                self.extract_rules_from_tree(tree)

    def build_feature_vectors(self) -> dict[str, float]:
        feature_vector: dict[str, float] = defaultdict(float)
        for lhs_symbol, rhs_counter in self.rules_counter.items():
            for rhs_symbols, freq in rhs_counter.items():
                rule = f"{lhs_symbol} -> {' '.join(rhs_symbols)}"
                dim = rule_to_dim.get(rule)
                if not dim:
                    continue
                feature_vector[dim] += freq
        total = sum(feature_vector.values()) or 1.0
        return {dim: freq / total for dim, freq in feature_vector.items()}

def load_and_extract(path: Path) -> dict[str, float]:
    extractor = PCFGExtractor()
    trees_data = extractor.load_trees(path)
    extractor.extract_from_data(trees_data)
    return extractor.build_feature_vectors()

def save_vector_for_character(character: str, vector: dict[str, float]) -> None:
    stored = False
    for item in dataset_storage.items.values():
        if item["character"] == character:
            dataset_storage.save_component(item["output"], syntactic_vector=vector)
            stored = True
        elif stored:
            break
    if not stored:
        raise ValueError(f"角色 {character} 的 PCFG 特征未匹配到任何句子")

pcfg_muice_vector = load_and_extract(Path("./outputs/cons/muice_cons.json"))
pcfg_haruhi_vector = load_and_extract(Path("./outputs/cons/haruhi_cons.json"))
pcfg_psydc_vector = load_and_extract(Path("./outputs/cons/psydc_cons.json"))

save_vector_for_character("Muice", pcfg_muice_vector)
save_vector_for_character("Ayaka", pcfg_haruhi_vector)
print("成功构建并保存 PCFG 特征向量")

成功构建并保存 PCFG 特征向量


## 处理语用风格层向量

### 读取各语料库语用风格文件

In [48]:
Pragmatic_Muice = "./outputs/pragmatic/muice.jsonl"
Pragmatic_ayaka = "./outputs/pragmatic/ayaka.jsonl"

class RawPCFGItem(TypedDict):
    prompt: str
    response: str
    pragmatic_styles: list[dict[str, float]]

class PCFGItem(TypedDict):
    response: str
    pragmatic_styles: list[str]

def read_pcfg_jsonl_file(jsonl_file: Path, threshold: Optional[float] = None) -> list[PCFGItem]:
    with open(jsonl_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    raw_items: list[RawPCFGItem] = []
    items: list[PCFGItem] = []

    for line in lines:
        if line := line.strip():
            raw_item: RawPCFGItem = json.loads(line)
            raw_items.append(raw_item)

    # list[dict[str, float]] -> dict[str, float] -> list[str]
    for raw_item in raw_items:
        raw_pragmatic_styles = raw_item["pragmatic_styles"]
        pragmatic_styles: dict[str, float] = {}

        for vec in raw_pragmatic_styles:
            pragmatic_styles.update(vec)

        threshold = threshold or 0
        final_styles: list[str] = []

        for key, value in pragmatic_styles.items():
            if value > threshold:
                final_styles.append(key)
        
        item = PCFGItem(response=raw_item["response"], pragmatic_styles=final_styles)
        items.append(item)

    return items

pcfg_muice_items = read_pcfg_jsonl_file(Path(Pragmatic_Muice), 0.4)
pcfg_ayaka_items = read_pcfg_jsonl_file(Path(Pragmatic_ayaka), 0.4)

pcfg_import_items = pcfg_muice_items + pcfg_ayaka_items

skiped = 0
for item in pcfg_import_items:
    try:
        dataset_storage.save_component(item["response"], pragmatic_styles=item["pragmatic_styles"])
    except ValueError as e:
        skiped += 1
        continue

print(f"更新了 {len(pcfg_import_items)} 条训练集条目的风格向量，已跳过: {skiped} 条")


更新了 4918 条训练集条目的风格向量，已跳过: 714 条


## 导出训练集文件

In [49]:
OUTPUT_PATH = Path("./dataset/llm_train.json")

dataset_storage.output(OUTPUT_PATH)

训练集已导出至 dataset\llm_train.json, 总有效训练集数量: 3565, 跳过数量: 1019


## 定义模型结构&数据加载器

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import json
from typing_extensions import TypedDict
from transformers.data.data_collator import DataCollatorWithPadding
from pathlib import Path
from transformers import Qwen3ForCausalLM

class InstructionComponents(TypedDict):
    lexical_keywords: list[str]
    syntactic_vector: dict[str, float]
    pragmatic_styles: list[str]

class DatasetItem(TypedDict):
    character: str
    neutral_sentence: str
    instruction_components: InstructionComponents
    output: str

class StyleDataset(Dataset):
    def __init__(self, path:Path, tokenizer, prag_style_vocab: list[str], syntactic_dims: list[str]):
        self.data: list[DatasetItem] = json.loads(path.read_text(encoding="utf-8"))
        self.tokenizer = tokenizer
        self.prag_vocab = {style: i for i, style in enumerate(prag_style_vocab)}
        self.syntactic_dims = syntactic_dims

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx:int):
        item = self.data[idx]
        instr = item["instruction_components"]

        # --- (1) 中性句 token 化 ---
        enc = self.tokenizer(
            item["neutral_sentence"],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        # --- (2) 句法向量 (float tensor) ---
        syntactic_vec = torch.tensor(
            [instr["syntactic_vector"].get(dim, 0.0) for dim in self.syntactic_dims],
            dtype=torch.float
        )

        # --- (3) 语用风格向量 (multi-hot) ---
        prag_vec = torch.zeros(len(self.prag_vocab))
        for tag in instr["pragmatic_styles"]:
            if tag in self.prag_vocab:
                prag_vec[self.prag_vocab[tag]] = 1.0

        # --- (4) 输出目标句 ---
        with self.tokenizer.as_target_tokenizer():
            label_enc = self.tokenizer(item["output"], truncation=True, padding="max_length", max_length=128, return_tensors="pt")

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "syntactic_vec": syntactic_vec,
            "prag_vec": prag_vec,
            "labels": label_enc["input_ids"].squeeze(0)
        }

class StyleDataCollator(DataCollatorWithPadding):
    """动态 padding + 风格特征拼接"""

    def __call__(self, features):
        syntactic_vecs = torch.stack([f.pop("syntactic_vec") for f in features])
        prag_vecs = torch.stack([f.pop("prag_vec") for f in features])

        batch = super().__call__(features)  # 动态 padding + mask + label对齐

        # 把风格特征重新放回 batch
        batch["syntactic_vec"] = syntactic_vecs
        batch["prag_vec"] = prag_vecs
        return batch

class StyleEncoder(nn.Module):
    def __init__(self, syntactic_dim, prag_style_size, prag_emb_dim=32, out_dim=128):
        super().__init__()
        self.prag_emb = nn.Sequential(
            nn.Linear(prag_style_size, prag_emb_dim),
            nn.ReLU(),
            nn.LayerNorm(prag_emb_dim)
        )
        self.proj = nn.Sequential(
            nn.Linear(syntactic_dim + prag_emb_dim, out_dim),
            nn.Tanh()
        )

    def forward(self, syntactic_vec, prag_vec):
        prag_emb = self.prag_emb(prag_vec)
        s_raw = torch.cat([syntactic_vec, prag_emb], dim=-1)
        return self.proj(s_raw)  # [batch, out_dim]
    
class ConditionalAdapter(nn.Module):
    def __init__(self, input_dim, bottleneck_dim, style_dim):
        super().__init__()
        self.down = nn.Linear(input_dim, bottleneck_dim)
        self.act = nn.ReLU()
        self.up = nn.Linear(bottleneck_dim, input_dim)
        self.style_gate = nn.Sequential(
            nn.Linear(style_dim, style_dim * 2),
            nn.ReLU(),
            nn.Linear(style_dim * 2, 2 * input_dim)
        )

    def forward(self, hidden_states, style_vector):
        adapter_out = self.up(self.act(self.down(hidden_states)))
        gamma, beta = self.style_gate(style_vector).chunk(2, dim=-1)
        gamma, beta = gamma.unsqueeze(1), beta.unsqueeze(1)
        return hidden_states + adapter_out * gamma + beta
    
class StyledModel(nn.Module):
    def __init__(self, base_model:Qwen3ForCausalLM, style_dim, bottleneck_dim=64):
        super().__init__()
        self.model = base_model
        hidden_size = base_model.config.hidden_size

        # Freeze the base model
        for p in self.model.parameters():
            p.requires_grad = False

        num_layers = len(base_model.model.layers)
        self.adapters = nn.ModuleList([
            ConditionalAdapter(hidden_size, bottleneck_dim, style_dim)
            for _ in range(num_layers)
        ])

        # Store the current style vector using a placeholder
        self.current_style_vector = None

        # Register a forward hook for each transformer block
        for i, layer in enumerate(self.model.model.layers):
            layer.register_forward_hook(self.create_hook(i))

    def create_hook(self, layer_idx):
        # This closure captures the layer_idx
        def hook(module, input, output):
            # The output of a transformer block is a tuple, hidden_state is the first element
            hidden_state = output[0]
            # Apply the corresponding adapter
            styled_hidden_state = self.adapters[layer_idx](hidden_state, self.current_style_vector)
            # Return the modified output
            return (styled_hidden_state,) + output[1:]
        return hook

    def forward(self, input_ids, attention_mask=None, labels=None, style_vector=None):
        # Store the style vector so the hooks can access it
        self.current_style_vector = style_vector

        # The hooks will automatically be called during this forward pass
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels, # Pass labels directly to the base model for a cleaner loss calculation
        )

        # Clear the stored vector to avoid memory leaks
        self.current_style_vector = None

        # The base model can often compute the loss for you if labels are provided
        return {"loss": outputs.loss, "logits": outputs.logits}

c:\Users\Muika\.conda\envs\Paper\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 加载模型&训练数据

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.optimization import get_scheduler
from torch.utils.data import DataLoader

MODEL_PATH = "../Models/Qwen3-4B-Instruct-2507"
DATASET_PATH = Path("./dataset/llm_train.json")
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
EPOCH = 3

# 定义风格向量信息

syntactic_dims = ['declarativity', 'prepositional_density', 'interjectionality', 'parallelism', 'nominal_coordination', 'referentiality',
                  'modifier_density', 'subordination']
prag_style_vocab = ['kind', 'modest', 'clingy', 'playful', 'cold', 'proud', 'sharp_tongued', 'subservient', 'submissive', 'controlling',
                    'strong', 'defensive', 'tsukkomi', 'rational', 'curious', 'imaginative', 'cautious', 'idealistic', 'conservative',
                    'radical', 'obsessive', 'hesitant', 'energetic', 'optimistic', 'confident', 'passionate', 'melancholy', 'serious',
                    'emotional', 'sensitive', 'shy', 'irritable', 'anxious', 'lazy', 'tsundere', 'yandere', 'chuunibyou', 'cute', 'naive',
                    'airhead', 'elegant', 'humorous', 'loyal', 'responsible', 'willful', 'antisocial', 'talkative', 'masochistic', 'sadistic', 'evil']

# 加载模型和训练数据

print("Loading Model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
print(f"Successfully loaded {MODEL_PATH}")

print("Loading Dataset...")
dataset = StyleDataset(DATASET_PATH, tokenizer, prag_style_vocab, syntactic_dims)
collator = StyleDataCollator(tokenizer, pad_to_multiple_of=8)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collator)
print(f"Successfully loaded dataset, length: {len(dataset)}")

print("Preparing for train...")
style_encoder = StyleEncoder(len(syntactic_dims), len(prag_style_vocab), out_dim=128)
styled_model = StyledModel(base_model, style_dim=128)
optimizer = torch.optim.AdamW(list(styled_model.parameters()) + list(style_encoder.parameters()), lr=LEARNING_RATE)

num_training_steps = len(dataloader) * EPOCH

lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
styled_model.to(device)
style_encoder.to(device)


Loading Model...


Loading checkpoint shards: 100%|██████████| 3/3 [00:10<00:00,  3.41s/it]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Successfully loaded ../Models/Qwen3-4B-Instruct-2507
Loading Dataset...
Successfully loaded dataset, length: 3565
Preparing for train...


NotImplementedError: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.

## 训练模型

In [ ]:
from tqdm import tqdm

styled_model.train()

for epoch in range(EPOCH):
    loop = tqdm(dataloader, desc=f"Epoch {epoch+1}")
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        syn_vec = batch["syntactic_vec"].to(device)
        prag_vec = batch["prag_vec"].to(device)

        style_vec = style_encoder(syn_vec, prag_vec)
        outputs = styled_model(input_ids, attn, labels=labels, style_vector=style_vec)

        loss = outputs["loss"]
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        loop.set_postfix(loss=loss.item())

In [4]:
print(base_model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layernorm): Qwe